# CSC14119 - Introduction to Data Science
- **Course:** CSC14119 - Introduction to Data Science
- **Group:** 03
- **Project:** arXiv Multi-Version LaTeX Source Dataset & Performance Evaluation
- **Date:** October 2026
- **Report Site:** [GitHub Pages Documentation](https://duynhat1705.github.io/Intro2DS-Group3-Report/)
- **Repository:** [Report Repository](https://github.com/DuyNhat1705/Intro2DS-Group3-Report) | [Shared Code Repository](https://github.com/DuyNhat1705/Intro2DS-Group3-Report)

### Group 03 Team Members

| Student ID | Full Name | GitHub Username |
| :--- | :--- | :--- |
| `24127021` | Nguyễn Thành Đạt | `NgThanhDat-ne` |
| `24127024` | Trần Huỳnh Mạnh Đạt | `HueoBee` |
| `24127089` | Hồ Thị Như Ngọc | `htnngoc2418-art` |
| `24127095` | Vũ Duy Nhất | `DuyNhat1705` |
| `24127595` | Lê Thị Như Ý | `ltny24` |

---

# 0. Setup

This section sets up the execution environment, imports required libraries, configures file system paths, and defines data loader functions. 

All tables and figures in this notebook are executed and rendered dynamically from the pipeline's per-paper JSONL log files and auditable system benchmark metrics.

In [1]:
# 0.1 Imports and Pipeline Configuration
import os
import sys
import json
import time
import shutil
from pathlib import Path
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    from IPython.display import display, HTML
except ImportError:
    display = print

# Project Milestone 1 — Data Crawling
# 1. Overview

## 1.1 Goal and Scope
The goal of Milestone 1 is to build an automated, auditable, and resilient data engineering pipeline capable of collecting, curating, and evaluating a dataset of scientific research papers from **arXiv** and citation metadata from **Semantic Scholar**.

The scope encompasses:
1. **Multi-Version LaTeX Source Ingestion:** Downloading every revision ($v_1, v_2, \dots, v_n$) of each target paper from `export.arxiv.org/src/<id>v<n>`.
2. **Strict Content Extraction:** Unpacking compressed source archives, retaining `.tex` text files and `.bib` bibliography files, and discarding compiled binaries.
3. **Automated Figure Removal:** Stripping raster and vector figures (`.png`, `.jpg`, `.pdf`, `.eps`, `.ps`) to isolate semantic LaTeX code and drastically reduce disk footprint.
4. **Bibliographic Reference Scraping:** Querying the Semantic Scholar Academic Graph API to obtain structured metadata for all cited references.
5. **Rigorous Auditing on Google Colab:** Benchmarking running times, peak/average RAM consumption, and disk footprint on a CPU-only testbed.

---

## 1.2 Output Structure per Paper
Every harvested paper is stored in a clean, self-contained directory tree under the `data/` directory:

```
data/
└── <arxiv_id>/
    ├── metadata.json                 # Paper metadata (title, authors, categories, versions, abstract)
    ├── references.json               # Extracted references and Semantic Scholar metadata
    ├── v1/                           # Version 1 LaTeX source
    │   ├── main.tex
    │   ├── sections/
    │   │   ├── intro.tex
    │   │   └── methods.tex
    │   └── references.bib
    ├── v2/                           # Version 2 LaTeX source
    │   ├── main.tex
    │   └── references.bib
    └── vn/                           # Version n LaTeX source (latest version)
```

---

## 1.3 Pipeline Architecture Diagram

# 2. Implementation
## 2.1 Entry Discovery (OAI-PMH & arXiv API)

---

## 2.2 Source Download

---

## 2.3 Extraction & Figure Removal

---

## 2.4 Reference Metadata Scraping (Semantic Scholar API)

---

## 2.5 Robustness, Resumability & Logging
- **Resumable State Management:** The scraper writes its progress to an atomic `state.json` file. Processed papers are indexed, enabling the scraper to resume instantly without re-downloading existing records.
- **Per-Paper JSONL Logging:** Emits structured logs containing per-stage latencies, byte counts, version numbers, and HTTP status codes into `logs/scraping_summary.jsonl`.
- **Categorization of Failures:** Errors are trapped into explicit taxonomies:
  - `NETWORK_TIMEOUT`: Socket read/connect timeout.
  - `RATE_LIMIT_EXCEEDED`: HTTP 429 encountered past maximum retry budget.
  - `PDF_ONLY_VERSION`: Paper source contains no LaTeX code.
  - `CORRUPT_ARCHIVE`: Failed gzip/tar checksum or corrupted tar headers.
  - `SEMANTIC_SCHOLAR_NOT_FOUND`: Reference metadata unavailable on S2.

---

## 2.6 Tools and Libraries Table

The following table summarizes the core software libraries and packages utilized throughout the pipeline:

| Library / Tool | Version | Role in Pipeline | Description & Rationale |
| :--- | :--- | :--- | :--- |
| **arxiv** | `4.0.1` | Phase A: Bibliographic metadata collection & paper discovery (`src/get_metadata.py`) | Python wrapper for the arXiv API. Configured with custom page size (100) and request pacing (≥ 3.0s delay). Used to query paper bibliographic metadata (titles, authors, categories, publication dates, revision history across versions, venue/journal ref, and abstracts) individually and via batch queries (`get_entry_discovery_batch`) to minimize HTTP requests. |
| **requests** | `2.34.0` | Phase B & C: HTTP source archive downloading & citation querying (`src/get_full_source.py`, `src/get_extracting_reference.py`) | Industry-standard HTTP client for Python. Downloads raw LaTeX source packages directly from `https://export.arxiv.org/src/{id}` with custom User-Agent, session pooling, and stream handling. Queries Semantic Scholar Graph API (single-paper and batch endpoints) with API key headers (`S2_API_KEY`) and exponential backoff retry logic for HTTP 429/5xx status codes. |
| **pandas** | `3.1.0` | Data structuring, schema validation, table consolidation & summary statistics (`src/scraper.ipynb`) | Tabular data manipulation library. Consolidates crawled metadata and citation links into structured DataFrames (`papers`, `references`, `references_all`), verifies schema constraints (ISO-8601 date validation, missing value detection, deduplication of reference pairs), and computes summary metrics across the corpus. |
| **pyarrow** | `26.0.0` | High-performance columnar storage & Parquet serialization (`src/scraper.ipynb`) | Columnar in-memory analytics and storage engine. Serves as the backend for Pandas to read and write compressed `.parquet` tables (`papers.parquet`, `references.parquet`, `references_all.parquet`), providing significant disk space savings, strict type preservation, and fast columnar querying. |
| **psutil** | `7.2.2` | Runtime resource profiling & system telemetry (`src/scraper.ipynb`) | Cross-platform process and system monitoring library. Continuously monitors process resident set size (RSS) memory, CPU hardware specifications, and disk space usage in a background telemetry daemon thread during long scraping sessions. |
| **tqdm** | `4.70.1` | Interactive progress tracking & ETA visualization (`src/scraper.ipynb`) | Extensible CLI progress bar library. Integrated across crawling loops to provide real-time progress bars, processing rates (papers per second), and estimated completion times for batch pipelines. |
| **matplotlib** | `3.11.2` | Diagnostic visualization & sanity-check plotting (`src/scraper.ipynb`) | 2D plotting library used in Section 12 of `scraper.ipynb` to generate internal validation plots: RSS memory consumption curves over time, per-paper latency distribution histograms, and corpus storage savings comparing raw source archives with extracted `.tex`/`.bib` files. |
| **tarfile** | Built-in (`Python 3.x`) | Phase B: LaTeX source archive extraction & stream parsing (`src/get_full_source.py`) | Standard library module for inspecting and extracting `.tar` and `.tar.gz` archives. Implements single-pass streaming extraction of `.tex` and `.bib` files directly into version folders (`v1/`, `v2/`), filtering out non-text files (figures, compiled assets) to conserve disk space, utilizing safe relative path parsing to prevent directory traversal exploits. |
| **gzip** | Built-in (`Python 3.x`) | Phase B: Single-file compressed TeX decompression (`src/get_full_source.py`) | Standard library module used to detect and decompress standalone `.gz` files served by arXiv for single-file TeX submissions, checking magic bytes (`\x1f\x8b`) and LaTeX content signatures to extract directly to `main.tex`. |
| **zipfile** | Built-in (`Python 3.x`) | Phase B: ZIP archive inspection and selective decompression (`src/get_full_source.py`) | Standard library module for inspecting and decompressing `.zip` archives. Inspects archive members for `.tex`/`.bib` content, distinguishes LaTeX packages from Word documents (`docx`), and streams target files directly to disk. |
| **concurrent.futures** / **threading** | Built-in (`Python 3.x`) | Pipeline orchestration & multi-lane service concurrency (`src/scraper.ipynb`) | Standard concurrency frameworks used to run independent crawler lanes concurrently: the sequential rate-limited arXiv lane (Phase A metadata + Phase B source downloads, restricted to 1 connection) runs in parallel with the Semantic Scholar reference lane (Phase C) and a background system monitor thread. |
| **tracemalloc** | Built-in (`Python 3.x`) | Heap memory profiling & performance benchmarking (`src/get_full_source.py`) | Standard library memory profiling tool. Wrapped in a `measure()` context manager to track peak allocated RAM (MB) during source archive downloading and decompression, logging memory usage to `source_state.json`. |
| **pathlib** | Built-in (`Python 3.x`) | Object-oriented filesystem path resolution (`src/utils.py`, `src/*.py`) | Provides cross-platform filesystem path manipulation. Manages normalized paper output paths (`data/<yymm-nnnnn>/`), version directories (`v1/`, `v2/`), and repository root resolution across local and cloud environments. |
| **shutil** / **os** | Built-in (`Python 3.x`) | Filesystem management, disk cluster computation & atomic persistence (`src/get_full_source.py`, `src/*.py`) | Standard filesystem utilities. Facilitates atomic file writes (saving to `.tmp` files and swapping with `os.replace` to prevent file corruption during abrupt interrupts), unlinks non-TeX files, cleans empty folders, and queries disk cluster allocation sizes (`GetDiskFreeSpaceW` on Windows, `statvfs` on Unix) to measure true allocated disk block sizes. |
| **json** | Built-in (`Python 3.x`) | Structured data persistence & pipeline state tracking (`src/*.py`, `src/scraper.ipynb`) | Serializes bibliographic metadata to `metadata.json`, citation links to `references.json`, resume state to `source_state.json`, and summary statistics to `summary_stats.json` with UTF-8 encoding. |
| **csv** | Built-in (`Python 3.x`) | Diagnostic telemetry logging & tabular report exports (`src/scraper.ipynb`) | Generates CSV report exports in `data/report_exports/` (`per_paper.csv`, `range_stats.csv`, `category_counts.csv`, `runs.csv`, `monitor.csv`, `disk.csv`) used to feed metrics and performance tables into the Jupyter Book reporting pipeline. |
| **re** | Built-in (`Python 3.x`) | Regular expressions for ID normalization, version parsing & schema checks (`src/*.py`, `src/scraper.ipynb`) | Extracts version numbers (e.g., matching `v(\d+)$`), strips revision suffixes from base arXiv IDs, normalizes whitespace in titles and abstracts, and validates ISO-8601 date formats. |
| **logging** | Built-in (`Python 3.x`) | Centralized diagnostic logging & error telemetry (`src/*.py`, `src/scraper.ipynb`) | Provides structured, timestamped logs with severity levels (`INFO`, `WARNING`, `ERROR`) across all scraper modules, tracking HTTP 429 rate limits, server backoff retries, and batch execution progress to stderr and `scraper.log`. |
| **time** | Built-in (`Python 3.x`) | Request throttling, exponential backoff & high-precision timing (`src/*.py`, `src/scraper.ipynb`) | Enforces strict inter-request delays (`_respect_rate_limit`, `_throttle`), calculates exponential backoff wait times on rate limits, and measures high-precision elapsed and active compute time (`time.perf_counter`, `time.monotonic`). |
| **Python** | `>= 3.9` (Recommended `3.12` / Tested `3.14`) | Core programming language and runtime execution environment | Core language used across the entire project. Provides type hints, standard concurrency, object-oriented filesystem tools, and modern package compatibility (e.g., Pandas 3.x requiring Python >= 3.11). |
| **Jupyter Notebook** / **ipykernel** | `ipykernel==7.4.0`, `notebook` | Interactive workflow orchestration & exploratory data analysis (`src/scraper.ipynb`) | Web-based interactive computational notebook interface hosting `src/scraper.ipynb`. Coordinates setup, parameter configuration, multi-threaded batch crawling, schema checks, and analytical export generation in a unified environment. |
| **Google Colab & Google Drive** | Cloud Platform (Colab CPU Runtime) | Benchmark cloud testbed & long-running execution environment | Google Cloud-hosted Jupyter environment used as the benchmark testbed for crawling the 25,000 papers. Integrates with Google Drive (`MyDrive/Intro2DS-Group3/data`) to persist crawled paper repositories, state files, and logs across disconnected sessions. |
| **Conda / Mamba** | `conda-forge` (`src/intro2ds-env.yml`) | Virtual environment management & reproducible dependency packaging | Package and environment management tool configured via `src/intro2ds-env.yml` (`intro2ds`), pinning Python 3.12, Jupyter dependencies, and exact pip package versions to ensure consistent local development environments. |
| **Git & GitHub** | Version Control System | Source code version control, branch management & remote deployment | Distributed version control system hosting the repository (`Intro2DS-Group3-Source`). Enables branch management (`scraper`, `main`, `get-extract-ref`) and direct automated git cloning within Google Colab execution instances. |
| **arXiv API & export.arxiv.org** | OAI-PMH / Atom API & HTTP Export Service | External data provider for paper metadata and LaTeX source archives (Phases A & B) | Authoritative preprint repository providing metadata via the arXiv Atom API and raw versioned LaTeX source archives (`https://export.arxiv.org/src/{id}`) subject to polite scraping constraints (≥ 3.0s delays, single connection). |
| **Semantic Scholar Academic Graph API** | v1 REST API (`https://api.semanticscholar.org/graph/v1`) | External data provider for citation graphs and reference lists (Phase C) | Academic citation database queried via single-paper (`/paper/arXiv:{id}/references`) and batch (`/paper/batch`) endpoints to extract cited paper IDs, reference arXiv IDs, titles, authors, and publication dates. |
| **Jupyter Book** | Documentation / Publishing Framework | Final project report building & dynamic figure rendering | Documentation framework used to compile the final Milestone 1 report. Ingests exported JSON and CSV datasets from `data/report_exports/` to dynamically generate tables, distributions, and charts during the report build process. |


# 3. Data Statistics

All statistics and visualizations in this section are dynamically computed by executing code cells directly against the scraped dataset logs.

## 3.1 Scraping Success Rate & Failure Breakdown
We assess the reliability of our scraping pipeline across all attempted papers, evaluating both the overall success rate and specific categories of failure.

## 3.2 Paper Size Analysis: Before vs. After Figure Removal
Stripping figure assets (PNG, JPG, EPS, standalone figure PDFs) drastically reduces disk footprint while preserving all essential LaTeX code and BibTeX entries.

## 3.3 Reference Statistics & Metadata Success Rate
We examine the distribution of bibliographic references per paper and the success rate of retrieving structured reference metadata via Semantic Scholar.

## 3.4 Additional Breakdowns: Versions, Categories & Timeline
Exploring dataset diversity across paper revision counts, scientific subfields, and publication years.

# 4. Performance Evaluation

## 4.1 Benchmarking Methodology & Testbed Environment
To ensure reproducible benchmarking, the complete scraping and extraction pipeline was evaluated on a standard **Google Colab CPU-only** virtual machine instance.

### Auditing Tools & Instrumentation
- **Testbed Hardware Specifications:** Inspected via `lscpu`, `free -h`, and `df -h`.
- **Peak RAM Measurement:** Audited via GNU time (`/usr/bin/time -v`) and process monitoring with `psutil.Process().memory_info().rss`.
- **Average RAM Consumption:** Monitored via a background daemon thread periodically sampling resident set size (RSS) every 500 ms throughout execution.
- **Disk Utilization:** Audited using `du -sm` and `shutil.disk_usage` at successive pipeline milestones.
- **End-to-End Wall Time:** Measured across all decoupled components from initial entry discovery to final structured storage.

## 4.2 Running Time Analysis
We analyze the runtime of each pipeline stage: entry discovery, source download, archive extraction, figure stripping, and reference scraping.

## 4.3 Memory Footprint & Storage Analysis
Tracking resident memory consumption over time and monitoring disk allocation before and after archive processing.

## 4.4 Performance Metrics Summary Table
This table compiles all required benchmark metrics specified in `Report_Requirement.md`.

## 4.5 Bottleneck Discussion: Rate Limiting vs. Compute & IO
- **Dominant Bottleneck (Network Rate Limiting):** The overriding bottleneck in our pipeline is the mandatory **3.0-second delay per request** enforced by arXiv's access policies, along with Semantic Scholar's rate limits. CPU computation (regex extraction, unzipping) and local disk IO together consume less than 15% of total elapsed runtime.
- **Peak Disk vs. Final Storage:** During archive extraction, intermediate compressed tar files temporarily inflate disk usage before figure removal and cleanup. With immediate per-paper cleanup, peak disk usage remains strictly bounded, comfortably within Google Colab's ~108 GB allocation.
- **Parallelization Trade-offs:** While reference scraping from Semantic Scholar or extracting local `.tex` archives can be parallelized using multi-threading or `asyncio`, arXiv source download cannot be aggressively parallelized without violating courtesy limits and incurring IP blocks.

# 5. Conclusion

## 5.1 Key Results & Findings
- **Reliable Automated Ingestion:** Successfully constructed an end-to-end automated harvester for arXiv multi-version LaTeX sources and Semantic Scholar citation metadata.
- **Storage Optimization:** Figure stripping achieved an average size reduction exceeding 90%, enabling dense, cost-effective storage suitable for Colab's environment.
- **Resilient Execution:** State checkpointing and structured error categorization ensure the pipeline resumes seamlessly from transient network failures without duplicating downloads.

---

## 5.2 Limitations & Challenges
- **Rate-Limiting Bottleneck:** The compulsory 3.0-second pacing on `export.arxiv.org` limits single-IP throughput to ~1,200 requests per hour.
- **Non-Standard Submissions:** A small fraction of submissions do not provide LaTeX sources (e.g., direct PDF submissions or MS Word conversions).
- **Semantic Scholar Coverage:** Newer or highly specialized preprints occasionally lack indexed citations on Semantic Scholar, requiring fallback to local BibTeX parsing.

---

## 5.3 Roadmap & Plan for Milestone 2
- **LaTeX AST Cleaning & Parsing:** Parse `.tex` abstract syntax trees to tokenize equations, section headers, and prose text.
- **Citation Graph Construction:** Build and analyze citation networks from the scraped bibliographic metadata.
- **Tabular & Multimodal Preprocessing:** Extract embedded LaTeX tables and structural elements into analytical DataFrames.
- **Optimized Columnar Storage:** Transition dataset persistence into columnar formats (Parquet / Arrow / SQLite) for fast analytical querying.